In [3]:
import numpy as np
from PIL import Image
import imageio
import math

# -----------------------------
# Forward Diffusion Utilities
# -----------------------------

def linear_beta_schedule(timesteps, start=1e-4, end=0.02):
    """
    Linear noise schedule used in DDPM.
    """
    return np.linspace(start, end, timesteps)

def forward_diffusion_sample(x0, t, sqrt_alphas_cumprod, sqrt_one_minus_alphas_cumprod):
    """
    q(x_t | x_0) = sqrt(alpha_bar_t) * x0 + sqrt(1 - alpha_bar_t) * noise
    """
    noise = np.random.randn(*x0.shape)
    return sqrt_alphas_cumprod[t] * x0 + sqrt_one_minus_alphas_cumprod[t] * noise

# -----------------------------
# Load image and normalize
# -----------------------------
# Image should be grayscale or RGB. We'll convert to float32 in [0,1]
img = Image.open("Kelvin_Sleep.jpg").convert("RGB")
x0 = np.array(img).astype(np.float32) / 255.0

# -----------------------------
# Diffusion setup
# -----------------------------
T = 1000 # number of diffusion steps
betas = linear_beta_schedule(T)
alphas = 1.0 - betas
alphas_cumprod = np.cumprod(alphas)

sqrt_alphas_cumprod = np.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = np.sqrt(1 - alphas_cumprod)

# -----------------------------
# Produce frames
# -----------------------------
frames = []

for t in range(T):
    xt = forward_diffusion_sample(
        x0,
        t,
        sqrt_alphas_cumprod,
        sqrt_one_minus_alphas_cumprod
    )
    # Clip + convert back to 0–255 uint8 for GIF
    xt_img = (np.clip(xt, 0, 1) * 255).astype(np.uint8)
    if t % 20 == 0:
        print(f"Generated frame for t={t}")
        frames.append(xt_img)

# -----------------------------
# Save GIF
# -----------------------------
imageio.mimsave("forward_diffusion.gif", frames, fps=20)

print("Saved forward_diffusion.gif")

Generated frame for t=0
Generated frame for t=20
Generated frame for t=40
Generated frame for t=60
Generated frame for t=80
Generated frame for t=100
Generated frame for t=120
Generated frame for t=140
Generated frame for t=160
Generated frame for t=180
Generated frame for t=200
Generated frame for t=220
Generated frame for t=240
Generated frame for t=260
Generated frame for t=280
Generated frame for t=300
Generated frame for t=320
Generated frame for t=340
Generated frame for t=360
Generated frame for t=380
Generated frame for t=400
Generated frame for t=420
Generated frame for t=440
Generated frame for t=460
Generated frame for t=480
Generated frame for t=500
Generated frame for t=520
Generated frame for t=540
Generated frame for t=560
Generated frame for t=580
Generated frame for t=600
Generated frame for t=620
Generated frame for t=640
Generated frame for t=660
Generated frame for t=680
Generated frame for t=700
Generated frame for t=720
Generated frame for t=740
Generated frame fo

In [4]:
from PIL import Image, ImageSequence

input_path = "forward_diffusion.gif"
output_path = "reverse_diffusion.gif"

# Load the original GIF
im = Image.open(input_path)

# Extract frames
frames = [frame.copy() for frame in ImageSequence.Iterator(im)]

# Reverse frame order
reversed_frames = list(reversed(frames))

# Save as a new GIF
reversed_frames[0].save(
    output_path,
    save_all=True,
    append_images=reversed_frames[1:],
    loop=0,
    duration=im.info.get("duration", 40),  # preserve timing if available
    disposal=2
)

print("Saved:", output_path)

Saved: reverse_diffusion.gif


In [5]:
from PIL import Image, ImageSequence

gif1_path = "forward_diffusion.gif"
gif2_path = "reverse_diffusion.gif"
output_path = "combined.gif"

# How long the delay should be (in milliseconds)
DELAY_MS = 1000   # adjust as needed

# Load GIFs
gif1 = Image.open(gif1_path)
gif2 = Image.open(gif2_path)

# Extract frames
frames1 = [f.copy() for f in ImageSequence.Iterator(gif1)]
frames2 = [f.copy() for f in ImageSequence.Iterator(gif2)]

# Create a delay frame (use last frame of GIF 1 for visual consistency)
delay_frame = frames1[-1].copy()

# We add a "duration" annotation to the delay frame
delay_frame.info["duration"] = DELAY_MS

# Combine all frames
combined_frames = frames1 + [delay_frame] + frames2

# Inherit duration from original gif (default if missing)
original_duration = gif1.info.get("duration", 40)

# Save as a combined GIF
combined_frames[0].save(
    output_path,
    save_all=True,
    append_images=combined_frames[1:],
    loop=0,
    duration=[original_duration] * len(frames1) 
             + [DELAY_MS] 
             + [gif2.info.get("duration", original_duration)] * len(frames2),
    disposal=2
)

print("Saved:", output_path)

Saved: combined.gif
